# buffer — Hugging Face model smoke test

Load and run a short generation for Qwen, Llama, and OLMo 3.

**Prerequisites**
- `uv sync` (includes torch, transformers, accelerate)
- GPU recommended (~16GB+ VRAM for 7–8B models)
- Llama is gated: `huggingface-cli login` and request access first

Run one model at a time if VRAM is tight — each test frees memory before the next.

In [1]:
import gc
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cpu


In [2]:
MODELS = {
    "qwen": "Qwen/Qwen2.5-0.5B-Instruct",
    "llama": "meta-llama/Llama-3.2-1B-Instruct",  # gated — needs HF login
    "olmo": "allenai/OLMo-2-0425-1B-Instruct",  # OLMo 2; OLMo 3 starts at 7B
}

PROMPT = "What is 2+2? Answer briefly."

In [3]:
def smoke_test(name: str, model_id: str, prompt: str = PROMPT, max_new_tokens: int = 32):
    """Load a model, generate once, print timing, then free memory."""
    print(f"\n{'=' * 60}")
    print(f"{name}: {model_id}")
    print('=' * 60)

    t0 = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto",
    )
    load_s = time.perf_counter() - t0
    print(f"Loaded in {load_s:.1f}s")

    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
    else:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    t1 = time.perf_counter()
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen_s = time.perf_counter() - t1

    new_tokens = outputs.shape[-1] - inputs["input_ids"].shape[-1]
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    print(f"Generation: {gen_s:.2f}s ({new_tokens} tokens, {gen_s / new_tokens * 1000:.0f} ms/token)")
    print(f"Response: {response!r}")

    del model, tokenizer, inputs, outputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"load_s": load_s, "gen_s": gen_s, "new_tokens": new_tokens, "response": response}

In [ ]:
results = {}
for name, model_id in MODELS.items():
    try:
        results[name] = smoke_test(name, model_id)
    except Exception as e:
        print(f"FAILED: {e}")
        results[name] = {"error": str(e)}

In [ ]:
# goal: check if tokenizer actually detokenizes each character
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
text = "L e t ' s   t h i n k"  # from a trigger line in your JSONL
print(tok.tokenize(text))
# or: tok.convert_ids_to_tokens(tok.encode(text))

'''
['L', 'Ġe', 'Ġt', "Ġ'", 'Ġs', 'ĠĠ', 'Ġt', 'Ġh', 'Ġi', 'Ġn', 'Ġk']
'''

['L', 'Ġe', 'Ġt', "Ġ'", 'Ġs', 'ĠĠ', 'Ġt', 'Ġh', 'Ġi', 'Ġn', 'Ġk']


'\n'

In [7]:
# Simple Qwen 0.5B smoke test.
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
)

messages = [{"role": "user", "content": "What is 2+2? Answer briefly."}]
result = generator(messages, max_new_tokens=16, do_sample=False)

print(result[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


2 + 2 equals 4.
